# clip_stacker — RIFE on Google Colab

Run the same **Practical-RIFE v4.26** pipeline that backs the Hugging Face Space
[`1inkusFace/RIFE`](https://huggingface.co/spaces/1inkusFace/RIFE)
(`huggingface/RIFE/` in [ford442/clip_stacker](https://github.com/ford442/clip_stacker)).

| Space / client contract | This notebook |
|---|---|
| `interpolate_video` | §3–5: single clip interpolate + optional boomerang |
| `batch_interpolate` | §6: a folder (e.g. Google Drive) or multi-file upload, one MP4 per input, resumable |
| multiplier `"2"` / `"4"` / `"8"` | same |
| output fps `"30"` / `"60"` / `"native"` | same meaning as `app.py` |
| `stitch` / `morph` | not here — those stay on the Space |

RIFE always generates `source_fps × multiplier` frames. `OUTPUT_FPS` only
controls the *encode*: `60` is the point of 4× on a 24 fps source; `30` matches
the Space's old default; `native` keeps every generated frame.

**Runtime → Change runtime type → T4 / L4 / A100 GPU.** CPU works but is slow.

Run §1 and §2 once, then either §3–5 (one clip) or §6 (batch).

In [ ]:
# GPU check — Runtime > Change runtime type > GPU if this says CPU
import torch, shutil
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("ffmpeg", shutil.which("ffmpeg"))
print("ffprobe", shutil.which("ffprobe"))

## 1. Install + clone Practical-RIFE + v4.26 weights

Same sources as `huggingface/RIFE/app.py` → `setup_environment()`:
- code: `https://github.com/hzwer/Practical-RIFE`
- weights: `https://huggingface.co/hzwer/RIFE/resolve/main/RIFEv4.26_0921.zip`

Two patches to upstream `inference_video.py` that Colab needs and the Space
gets another way:
- `scikit-video` still calls `np.float` / `np.int`, which NumPy ≥ 1.24 removed
  (Colab ships NumPy 2). The Space rewrites skvideo's sources; here a two-line
  alias shim goes in front of `import skvideo.io` instead, so nothing inside
  site-packages is touched.
- `transferAudio()` does `import moviepy.editor` and never uses it. moviepy 2.x
  removed `moviepy.editor`, and the import failing sends the script into an
  error path that crashes on a missing `_noaudio` file. The import is dropped.

In [ ]:
import os, re, shutil, subprocess, sys, zipfile
from pathlib import Path

# /content on Colab; the current directory anywhere else (local Jupyter).
BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
RIFE_DIR = BASE / "Practical-RIFE"
WORKSPACE = BASE / "workspace_temp"
MODEL_URL = "https://huggingface.co/hzwer/RIFE/resolve/main/RIFEv4.26_0921.zip"
STITCH_FPS = 30
VIDEO_TRACK_TIMESCALE = 30000

WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.check_call(cmd, shell=isinstance(cmd, str), **kw)

# Colab already has a CUDA torch; do not pip-install a CPU wheel over it.
# scikit-video is what inference_video.py reads frames with (skvideo.io.vreader).
run([sys.executable, "-m", "pip", "install", "-q",
     "scikit-video", "opencv-python>=4.1.2", "tqdm"])

if not (RIFE_DIR / ".git").exists():
    run(["git", "clone", "--depth", "1", "https://github.com/hzwer/Practical-RIFE", str(RIFE_DIR)])

hdv3 = RIFE_DIR / "HDv3"
train_log = RIFE_DIR / "train_log"
# Check the files themselves, not just the folders, so an interrupted download retries.
need_weights = not (hdv3 / "flownet.pkl").exists() or not (train_log / "RIFE_HDv3.py").exists()

if need_weights:
    zip_path = RIFE_DIR / "RIFEv4.26_0921.zip"
    run(["wget", "-q", "-O", str(zip_path), MODEL_URL])
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RIFE_DIR)
    extract = RIFE_DIR / "RIFEv4.26_0921"
    train_log.mkdir(exist_ok=True)
    for name in ("RIFE_HDv3.py", "IFNet_HDv3.py"):
        src = extract / name
        if src.exists():
            shutil.move(str(src), str(train_log / name))
    (train_log / "__init__.py").write_text("")
    hdv3.mkdir(exist_ok=True)
    flownet = extract / "flownet.pkl"
    if flownet.exists():
        shutil.move(str(flownet), str(hdv3 / "flownet.pkl"))
    shutil.rmtree(extract, ignore_errors=True)
    zip_path.unlink(missing_ok=True)

assert (hdv3 / "flownet.pkl").exists(), "RIFE weights missing — re-run this cell"
assert (train_log / "RIFE_HDv3.py").exists(), "RIFE model code missing — re-run this cell"

inf = RIFE_DIR / "inference_video.py"
text = inf.read_text()
patched = []

# Space patches mpeg4 → libx264. Current upstream writes through cv2.VideoWriter
# ('mp4v') and no longer contains this string, so it is usually a no-op; the
# final encode below re-encodes to H.264 either way.
old = "-c:v', 'mpeg4', '-qscale:v', '1'"
if old in text and "libx264" not in text:
    text = text.replace(old, "-c:v', 'libx264', '-preset', 'medium', '-crf', '23'")
    patched.append("codec → libx264")

NUMPY_SHIM = (
    "import numpy as _np  # clip_stacker: skvideo still uses np.float / np.int\n"
    "for _a, _t in (('float', float), ('int', int)):\n"
    "    if not hasattr(_np, _a):\n"
    "        setattr(_np, _a, _t)\n"
)
if "import skvideo.io\n" in text and NUMPY_SHIM not in text:
    text = text.replace("import skvideo.io\n", NUMPY_SHIM + "import skvideo.io\n", 1)
    patched.append("numpy alias shim")

if "    import moviepy.editor\n" in text:
    text = text.replace("    import moviepy.editor\n", "")
    patched.append("drop unused moviepy.editor import")

inf.write_text(text)
print("patched inference_video.py:", ", ".join(patched) or "nothing to do (already patched)")
print("RIFE_DIR", RIFE_DIR)
print("weights", sorted(p.name for p in hdv3.glob("*")))
print("train_log", sorted(p.name for p in train_log.glob("*.py")))

## 2. Helpers (same contract as `app.py`)

`nle_friendly_h264_video_args`, boomerang, probes and the RIFE timeout are
copied from `app.py` so Colab output matches what clip_stacker expects from the
Space. `interpolate_clip()` is `_interpolate_single()` without the GPU lease;
both the single-clip and the batch cells call it.

In [ ]:
import uuid

VIDEO_EXTENSIONS = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}

try:  # Colab-only widgets; everything else works in plain Jupyter too.
    from google.colab import files as colab_files, drive as colab_drive
except ImportError:
    colab_files = colab_drive = None

def nle_friendly_h264_video_args(gop=None, fps=None):
    is_native = fps == "native"
    fps = STITCH_FPS if fps is None or is_native else fps
    fps_val = float(fps)
    fps_str = str(int(fps_val)) if fps_val.is_integer() else str(fps_val)
    gop = gop or int(fps_val)
    args = [
        "-c:v", "libx264", "-preset", "fast", "-crf", "18",
        "-tune", "stillimage",
        "-profile:v", "high", "-level", "4.1",
        "-g", str(gop), "-keyint_min", str(gop), "-sc_threshold", "0",
        "-pix_fmt", "yuv420p",
        "-bf", "0",
    ]
    if not is_native:
        args += ["-vsync", "cfr", "-r", fps_str]
    args += [
        "-colorspace", "bt709", "-color_primaries", "bt709",
        "-color_trc", "bt709", "-color_range", "tv",
        "-video_track_timescale", str(VIDEO_TRACK_TIMESCALE),
    ]
    return args

def nle_friendly_aac_audio_args():
    return ["-c:a", "aac", "-b:a", "192k", "-ar", "44100", "-ac", "2"]

def get_fps(path):
    try:
        out = subprocess.check_output([
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "stream=avg_frame_rate",
            "-of", "default=noprint_wrappers=1:nokey=1", str(path),
        ], text=True, stderr=subprocess.DEVNULL).strip()
        if "/" in out:
            n, d = map(int, out.split("/"))
            return n / d if d else 30.0
        return float(out)
    except Exception:
        return 30.0

def get_duration(path):
    try:
        out = subprocess.check_output([
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1", str(path),
        ], text=True, stderr=subprocess.DEVNULL).strip()
        return float(out)
    except Exception:
        return None

def probe_has_audio(path):
    try:
        out = subprocess.check_output([
            "ffprobe", "-v", "error", "-select_streams", "a:0",
            "-show_entries", "stream=codec_name",
            "-of", "default=noprint_wrappers=1:nokey=1", str(path),
        ], text=True, stderr=subprocess.DEVNULL).strip()
        return bool(out)
    except Exception:
        return False

def rife_timeout_seconds(path, multi):
    duration = get_duration(path) or 60.0
    try:
        multi_val = max(1, int(str(multi).strip().replace("x", "") or "2"))
    except ValueError:
        multi_val = 2
    return max(300, int(duration * multi_val * 45 + 120))

def run_ffmpeg(cmd):
    """subprocess.run(check=True) that shows ffmpeg's own error, not just the exit code."""
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg failed ({r.returncode}):\n{r.stderr[-4000:]}")
    return r

def create_boomerang_loop(input_path, output_path, fps):
    sid = uuid.uuid4().hex
    rev = WORKSPACE / f"boom_rev_{sid}.mp4"
    fwd = WORKSPACE / f"boom_fwd_{sid}.mp4"
    lst = WORKSPACE / f"boom_list_{sid}.txt"
    enc = ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-r", str(fps),
           "-preset", "slow", "-crf", "17", "-an"]
    try:
        run_ffmpeg(["ffmpeg", "-i", str(input_path), "-vf", "reverse", *enc, "-y", str(rev)])
        run_ffmpeg(["ffmpeg", "-i", str(input_path), *enc, "-y", str(fwd)])
        lst.write_text(f"file '{fwd}'\nfile '{rev}'\n")
        run_ffmpeg(["ffmpeg", "-f", "concat", "-safe", "0", "-i", str(lst),
                    "-c", "copy", "-movflags", "+faststart", "-y", str(output_path)])
        return True
    except RuntimeError as e:
        print("Boomerang failed:", e)
        return False
    finally:
        for p in (rev, fwd, lst):
            p.unlink(missing_ok=True)

def run_rife_inference(video_path, output_path, multi):
    timeout = rife_timeout_seconds(video_path, multi)
    try:
        return subprocess.run(
            [sys.executable, "inference_video.py",
             "--video", str(video_path), "--output", str(output_path),
             "--multi", str(multi), "--model", "HDv3"],
            capture_output=True, text=True, cwd=str(RIFE_DIR), timeout=timeout,
        )
    except subprocess.TimeoutExpired as e:
        duration = get_duration(video_path)
        dur_str = f"{duration:.1f}s" if duration is not None else "unknown duration"
        raise RuntimeError(
            f"RIFE inference timed out after {timeout}s (input {dur_str}, {multi}x). "
            "Try a shorter clip or a lower multiplier."
        ) from e

def interpolate_clip(input_path, multi=4, output_fps="60", boomerang=False):
    """RIFE-interpolate one clip; returns the finished MP4 inside WORKSPACE.

    Mirrors app.py `_interpolate_single`: RIFE → NLE-friendly H.264 at
    `output_fps` (+ source audio, if any) → optional boomerang. Intermediates
    are deleted so a long batch does not fill the Colab disk.
    """
    factor = str(multi).strip().replace("x", "")
    fps_arg = str(output_fps).strip().lower()
    fps_arg = "native" if fps_arg == "native" else float(fps_arg)

    sid = uuid.uuid4().hex
    safe_input = WORKSPACE / f"input_{sid}{Path(input_path).suffix.lower() or '.mp4'}"
    rife_out = WORKSPACE / f"output_rife_{sid}.mp4"
    no_audio = WORKSPACE / f"output_rife_{sid}_noaudio.mp4"
    final_path = WORKSPACE / f"final_interp_{sid}.mp4"
    boom_path = WORKSPACE / f"final_boom_{sid}.mp4"

    # Copy to a plain name: inference_video.py shells out to ffmpeg with
    # unquoted paths, so spaces/quotes in the upload name would break it.
    shutil.copy(input_path, safe_input)
    try:
        r = run_rife_inference(safe_input, rife_out, factor)
        if r.returncode != 0:
            raise RuntimeError(f"RIFE failed:\n{r.stdout[-2000:]}\n{r.stderr[-4000:]}")

        src = rife_out if rife_out.exists() else no_audio
        if not src.exists():
            raise FileNotFoundError(f"RIFE wrote nothing. stdout:\n{r.stdout}\nstderr:\n{r.stderr}")

        cmd = ["ffmpeg", "-i", str(src)]
        if probe_has_audio(safe_input):
            cmd += ["-i", str(safe_input), "-map", "0:v:0", "-map", "1:a:0?",
                    *nle_friendly_h264_video_args(fps=fps_arg),
                    *nle_friendly_aac_audio_args(),
                    "-movflags", "+faststart", "-shortest", "-y", str(final_path)]
        else:
            cmd += [*nle_friendly_h264_video_args(fps=fps_arg),
                    "-an", "-movflags", "+faststart", "-y", str(final_path)]
        run_ffmpeg(cmd)
    finally:
        for stale in (safe_input, rife_out, no_audio):
            stale.unlink(missing_ok=True)

    if boomerang and create_boomerang_loop(final_path, boom_path, get_fps(final_path)):
        final_path.unlink(missing_ok=True)
        return boom_path
    return final_path

def ensure_drive(path):
    """Mount Google Drive if `path` lives on it and it is not mounted yet."""
    if str(path).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
        if colab_drive is None:
            raise RuntimeError("Google Drive paths only work on Colab")
        colab_drive.mount("/content/drive")

def describe(path):
    return (f"{get_duration(path)} s  fps: {round(get_fps(path), 3)}  "
            f"size_mb: {round(Path(path).stat().st_size / 1e6, 2)}")

print("helpers ready")

## 3. Single clip — upload

Use the file picker, or set `INPUT_PATH` to a file already on disk
(`/content/drive/MyDrive/...` mounts Drive automatically).

In [ ]:
# Leave empty to open the Colab upload widget
INPUT_PATH = ""

if INPUT_PATH:
    ensure_drive(INPUT_PATH)
else:
    if colab_files is None:
        raise SystemExit("Set INPUT_PATH (the upload widget only exists on Colab).")
    uploaded = colab_files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    name = next(iter(uploaded))
    dest = WORKSPACE / Path(name).name
    dest.write_bytes(uploaded[name])
    INPUT_PATH = str(dest)

assert Path(INPUT_PATH).is_file(), INPUT_PATH
print("input:", INPUT_PATH)
print("duration:", describe(INPUT_PATH))

## 4. Single clip — interpolate

Same knobs as the Space `interpolate_video` route / clip_stacker client.

In [ ]:
# --- knobs (match huggingface/RIFE README contract) ---
MULTI = 4            # 2 | 4 | 8
OUTPUT_FPS = "60"    # "30" | "60" | "native"
BOOMERANG = False
# ------------------------------------------------------

print(f"RIFE {MULTI}x → {OUTPUT_FPS}  timeout={rife_timeout_seconds(INPUT_PATH, MULTI)}s")
OUTPUT_PATH = str(interpolate_clip(INPUT_PATH, MULTI, OUTPUT_FPS, BOOMERANG))
print("output:", OUTPUT_PATH)
print("duration:", describe(OUTPUT_PATH))

## 5. Single clip — preview + download

In [ ]:
from IPython.display import Video, display

display(Video(OUTPUT_PATH, embed=True, width=640))
if colab_files is not None:
    colab_files.download(OUTPUT_PATH)

## 6. Batch

Interpolates every clip from a folder (or from a multi-file upload), one output
MP4 per input — the Colab counterpart of the Space's `batch_interpolate` route.
No stitching; each clip is processed independently.

- **`BATCH_INPUT_DIR`** — a folder of clips, e.g.
  `/content/drive/MyDrive/rife_in` (Drive is mounted automatically). Leave
  empty to pick several files in the upload widget instead.
- **`BATCH_OUTPUT_DIR`** — where results go. Point it at Drive so they survive
  a Colab disconnect. Default: `<input dir>_rife` for a folder, or
  `workspace_temp/batch_out` for uploads.
- Outputs are named `<source stem>_rife<MULTI>x_<fps>.mp4`, so re-running the
  cell with `SKIP_EXISTING = True` **resumes** where a disconnected session
  stopped. Results are written to a `.partial` file and renamed only when
  complete, so a half-copied file is never mistaken for a finished one.
- A clip that fails is reported and skipped; the rest of the batch carries on.

In [ ]:
# --- batch knobs -----------------------------------------------------------
BATCH_INPUT_DIR = ""       # folder of clips; "" = upload widget
BATCH_OUTPUT_DIR = ""      # "" = <input dir>_rife (folder) / workspace_temp/batch_out (upload)
BATCH_MULTI = 4            # 2 | 4 | 8
BATCH_OUTPUT_FPS = "60"    # "30" | "60" | "native"
BATCH_BOOMERANG = False
RECURSIVE = False          # also look in sub-folders of BATCH_INPUT_DIR
SKIP_EXISTING = True       # resume: skip clips whose output already exists
ZIP_AND_DOWNLOAD = False   # zip the outputs and download through the browser
# ---------------------------------------------------------------------------
import time

OUTPUT_TAG = re.compile(r"_rife\d+x_[\w.]+$")

def batch_output_name(src, multi, fps):
    fps_tag = str(fps).strip().lower()
    return f"{Path(src).stem}_rife{str(multi).strip().replace('x', '')}x_{fps_tag}.mp4"

def collect_batch_inputs(folder, recursive=False):
    folder = Path(folder)
    pattern = "**/*" if recursive else "*"
    clips = [p for p in folder.glob(pattern)
             if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
             and not p.name.startswith(".")
             and not OUTPUT_TAG.search(p.stem)]  # never re-interpolate our own outputs
    return sorted(clips, key=lambda p: str(p).lower())

if BATCH_INPUT_DIR:
    ensure_drive(BATCH_INPUT_DIR)
    in_dir = Path(BATCH_INPUT_DIR).resolve()
    assert in_dir.is_dir(), f"not a folder: {in_dir}"
    batch_inputs = collect_batch_inputs(in_dir, RECURSIVE)
    out_dir = Path(BATCH_OUTPUT_DIR) if BATCH_OUTPUT_DIR else in_dir.with_name(in_dir.name + "_rife")
else:
    if colab_files is None:
        raise SystemExit("Set BATCH_INPUT_DIR (the upload widget only exists on Colab).")
    up_dir = WORKSPACE / "batch_in"
    up_dir.mkdir(parents=True, exist_ok=True)
    uploaded = colab_files.upload()
    for name, data in uploaded.items():
        (up_dir / Path(name).name).write_bytes(data)
    # Only this upload's files — not leftovers from an earlier run.
    batch_inputs = sorted((up_dir / Path(n).name for n in uploaded
                           if Path(n).suffix.lower() in VIDEO_EXTENSIONS),
                          key=lambda p: str(p).lower())
    out_dir = Path(BATCH_OUTPUT_DIR) if BATCH_OUTPUT_DIR else WORKSPACE / "batch_out"

ensure_drive(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
if not batch_inputs:
    raise SystemExit(f"No video files ({', '.join(sorted(VIDEO_EXTENSIONS))}) found.")

# Unique output names even when two sources share a stem (a.mp4 + a.mov, or
# the same name in two sub-folders with RECURSIVE).
jobs, used = [], set()
for src in batch_inputs:
    name = batch_output_name(src, BATCH_MULTI, BATCH_OUTPUT_FPS)
    n = 2
    while name in used:
        name = batch_output_name(src.with_name(f"{src.stem}_{n}{src.suffix}"), BATCH_MULTI, BATCH_OUTPUT_FPS)
        n += 1
    used.add(name)
    jobs.append((src, out_dir / name))

print(f"{len(jobs)} clip(s) → {out_dir}   RIFE {BATCH_MULTI}x → {BATCH_OUTPUT_FPS}"
      + ("  + boomerang" if BATCH_BOOMERANG else ""))

done, skipped, failed = [], [], []
t_batch = time.time()
for i, (src, dest) in enumerate(jobs, 1):
    label = f"[{i}/{len(jobs)}] {src.name}"
    if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 0:
        print(f"{label}: skip (exists)")
        skipped.append(dest)
        continue
    t0 = time.time()
    print(f"{label}: {get_duration(src)} s @ {round(get_fps(src), 3)} fps …", flush=True)
    try:
        result = interpolate_clip(src, BATCH_MULTI, BATCH_OUTPUT_FPS, BATCH_BOOMERANG)
        partial = dest.with_name(dest.name + ".partial")
        shutil.copyfile(result, partial)  # copy (not move): WORKSPACE and Drive are different filesystems
        os.replace(partial, dest)
        Path(result).unlink(missing_ok=True)
        done.append(dest)
        print(f"{label}: ok in {time.time() - t0:.0f}s → {dest.name} ({describe(dest)})")
    except Exception as e:
        failed.append((src, e))
        print(f"{label}: FAILED after {time.time() - t0:.0f}s\n  {str(e)[-1500:]}")

print(f"\nbatch finished in {time.time() - t_batch:.0f}s: "
      f"{len(done)} done, {len(skipped)} skipped, {len(failed)} failed")
for src, e in failed:
    print("  failed:", src, "—", str(e).strip().splitlines()[-1] if str(e).strip() else type(e).__name__)

BATCH_OUTPUTS = [d for _, d in jobs if d.exists()]
if ZIP_AND_DOWNLOAD and BATCH_OUTPUTS and colab_files is not None:
    zip_path = WORKSPACE / f"rife_batch_{uuid.uuid4().hex[:8]}.zip"
    # MP4s are already compressed; ZIP_STORED just bundles them.
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
        for p in BATCH_OUTPUTS:
            z.write(p, p.name)
    print("zip:", zip_path, round(zip_path.stat().st_size / 1e6, 1), "MB")
    colab_files.download(str(zip_path))

## Notes

- Do **not** add a second RIFE implementation under `src/`. The Space at
  `huggingface/RIFE/` is the source of truth; this notebook is a Colab runner
  for the same weights + `inference_video.py` path.
- `@spaces.GPU` / Gradio / ZeroGPU token chunking do not apply on Colab: the
  batch simply runs clip after clip in one session. Each clip is its own
  `inference_video.py` process, so GPU memory is released between clips.
- Long clips + 8× will burn the Colab session. Start with a few seconds at 2×.
  For big batches write `BATCH_OUTPUT_DIR` to Drive and rely on `SKIP_EXISTING`
  to resume after a disconnect.
- If `inference_video.py` upstream changes flags, update the subprocess argv
  here and in `run_rife_inference()` in `huggingface/RIFE/app.py` together.